In [1]:
root = "C:/Users/v-wangwilli/projects/nepas"
setwd(root)
.libPaths(file.path(root, "renv/library/windows/R-4.4/x86_64-w64-mingw32"))

library(jsonlite)
library(tidyverse)

source("script/processors.R")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter()  masks stats::filter()
✖ purrr::flatten() masks jsonlite::flatten()
✖ dplyr::lag()     masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


# Manual cleaning steps

- Changed order of Nassau-Suffolk in 1972, page 18, line 66 to put OUTSIDE CENTRAL CITIES line last

In [2]:
years = 1967:1979

bps_cleaned = map(
  years,
  \(year) process_year(year, root = "data/cleaned_ocr/")
) |> bind_rows()

bps = add_smsa_indicators(bps_cleaned)

# Name cleaning
- corrected obvious typographical errors or inconsistencies
- took largest geographical area when possible --> combined multiple rows in earlier files
- aligned state abbreviations

To-do:
- fuzzy matching to see if didn't missed any inclusions

In [3]:
bps = bps |>
  mutate(
    smsa_name_clean = smsa_name |>
      str_remove_all("[0-9]|\\.|\\*|#") |>
      str_replace_all(" *- *", "-") |>
      str_squish() # Remove whitespace from ends and turn multiple spaces into one
  )

bps_smsa = bps |>
  filter(smsa_indicator == 1)

smsa_names = bps_smsa |>
  summarize(.by = smsa_name_clean, smsa_name = str_c(unique(smsa_name), collapse = " | "), line_number = first(line_number), year = first(year), page_numbers = first(page_numbers)) |>
  arrange(smsa_name_clean)

# write_csv(smsa_names, "derived_data/summaries/smsa_names.csv")

In [4]:
smsa_name_mapping = read_csv("derived_data/mappings/smsa_name_mapping.csv")

bps_corrected = bps_smsa |>
  left_join(smsa_name_mapping, join_by(smsa_name_clean)) |>
  summarize(
    .by = c(smsa_name_corrected, year),
    across(total_units:public_valuation, \(col) sum(col, na.rm = TRUE))
  ) |>
  separate_wider_delim(smsa_name_corrected, delim = ", ", names = c("central_city", "state"), cols_remove = FALSE)

states = bps_corrected |>
  distinct(state) |>
  arrange(state)

# write_csv(states, "derived_data/summaries/states.csv")

Rows: 377 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): smsa_name_clean, smsa_name_corrected

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
state_mapping = read_csv("derived_data/mappings/state_mapping.csv")
state_adoption = read_csv("derived_data/state_adoption_housing.csv")

bps_final = bps_corrected |>
  left_join(state_mapping, join_by(state)) |>
  mutate(states = state_corrected, primary_state = str_sub(state_corrected, 1, 2)) |>
  separate_longer_delim(state_corrected, delim = "-") |>
  left_join(state_adoption, join_by(state_corrected == state)) |>
  summarize(
    .by = c(smsa_name_corrected:public_valuation, states, primary_state),
    adoption_year = adoption_year |> replace_na(Inf) |> min(na.rm = TRUE)
  ) |>
  mutate(
    event_year = year - adoption_year,
    treat = as.numeric(event_year > -1)
  ) |>
  rename(smsa = smsa_name_corrected)

Rows: 81 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): state, state_corrected

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 7 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): state
dbl (1): adoption_year

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [6]:
head(bps_final)

write_csv(bps_final, "data/cleaned_ocr/bps_final.csv")

smsa,year,total_units,private_total,private_1_unit,private_2_units,private_3_4_units,private_5plus_units,public_units,private_structures,⋯,private_1_unit_val,private_2_units_val,private_3_4_units_val,private_5plus_units_val,public_valuation,states,primary_state,adoption_year,event_year,treat
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
"ABILENE, TEX",1967,247,247,95,2,0,150,0,6,⋯,1929,14,0,1380,0,TX,TX,Inf,-Inf,0
"AKRON, OHIO",1967,4732,4732,2289,540,159,1744,0,106,⋯,48831,5734,1567,18478,0,OH,OH,Inf,-Inf,0
"ALBANY, GA",1967,472,472,339,71,0,62,0,11,⋯,4627,278,0,430,0,GA,GA,1991,-24,0
"ALBANY-SCHENECTADY-TROY, NY",1967,3309,3309,1985,312,107,905,0,70,⋯,36164,3015,1260,6254,0,NY,NY,1975,-8,0
"ALBUQUERQUE, NMEX",1967,1334,1334,827,6,17,484,0,13,⋯,15703,43,117,3466,0,NM,NM,Inf,-Inf,0
"ALLENTOWN-BETHLEHEM-EASTON, PA-NJ",1967,3172,3172,1831,28,19,1294,0,83,⋯,32094,323,171,8284,0,PA-NJ,PA,Inf,-Inf,0
